# Humanoid — PPO with PyTorch

**Environment:** `Humanoid-v5` (MuJoCo 3D biped)  
**Algorithm:** Proximal Policy Optimization (PPO-Clip) with Gaussian MLP  

| Property | Detail |
|----------|--------|
| Observation | `(348,)` float64 — positions, velocities, inertias, forces |
| Action | `(17,)` float32 — joint torques ∈ [−0.4, 0.4] |
| Reward | healthy (5) + forward speed − ctrl cost − contact cost |
| Episode end | Torso height outside [1.0, 2.0] m, or 1000 steps |

## 1. Imports

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

print(f"Gymnasium : {gym.__version__}")
print(f"PyTorch   : {torch.__version__}")
DEVICE = torch.device("mps" if torch.backends.mps.is_available()
                       else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device    : {DEVICE}")

## 2. MLP Actor & Critic Networks

The observation is a **348-dimensional** continuous vector (positions, velocities,
inertias, contact forces), so a deep MLP is used.

| Network | Architecture | Output |
|---------|-------------|--------|
| **Actor** | 348 → 256 → 256 → mean (17) + log_std (17) | Gaussian policy |
| **Critic** | 348 → 256 → 256 → scalar | State value V(s) |

Actions are squashed with `tanh` then scaled to `[−0.4, 0.4]`.  
`Tanh` activations are used (better gradient flow vs ReLU for continuous control).

In [ ]:
N_OBS     = 348
N_ACTIONS = 17
ACT_LIMIT = 0.4    # joint torque bound
HIDDEN    = 256
LOG_STD_MIN, LOG_STD_MAX = -5, 2


class MLPActor(nn.Module):
    """Gaussian policy for continuous control."""

    def __init__(self, n_obs=N_OBS, n_actions=N_ACTIONS, hidden=HIDDEN, lr=3e-4):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(n_obs, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.mean_head    = nn.Linear(hidden, n_actions)
        self.log_std_head = nn.Linear(hidden, n_actions)
        self.optimizer    = optim.Adam(self.parameters(), lr=lr)

    def forward(self, obs):
        feat    = self.trunk(obs)
        mean    = torch.tanh(self.mean_head(feat)) * ACT_LIMIT
        log_std = self.log_std_head(feat).clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    @torch.no_grad()
    def select_action(self, obs_np):
        """Sample action; returns (action_np, detached log_prob)."""
        obs_t = torch.tensor(obs_np, dtype=torch.float32).to(DEVICE)
        mean, log_std = self.forward(obs_t)
        std  = log_std.exp()
        dist = torch.distributions.Normal(mean, std)
        raw  = dist.sample()
        # Tanh-scale squashing: a = tanh(raw) * ACT_LIMIT
        act  = torch.tanh(raw) * ACT_LIMIT
        log_prob = (dist.log_prob(raw)
                    - torch.log(ACT_LIMIT * (1 - torch.tanh(raw).pow(2)) + 1e-6)
                   ).sum(-1)
        return act.cpu().numpy(), log_prob.detach()

    def evaluate(self, obs_t, actions_t):
        mean, log_std = self.forward(obs_t)
        std  = log_std.exp()
        dist = torch.distributions.Normal(mean, std)
        raw  = torch.atanh((actions_t / ACT_LIMIT).clamp(-1 + 1e-6, 1 - 1e-6))
        log_prob = (dist.log_prob(raw)
                    - torch.log(ACT_LIMIT * (1 - (actions_t / ACT_LIMIT).pow(2)) + 1e-6)
                   ).sum(-1)
        entropy  = dist.entropy().sum(-1)
        return log_prob, entropy

    def learn(self, obs_t, actions_t, old_log_probs, advantages,
              epsilon=0.2, entropy_coef=0.0):
        new_log_probs, entropy = self.evaluate(obs_t, actions_t)
        ratio  = torch.exp(new_log_probs - old_log_probs)
        surr1  = ratio * advantages
        surr2  = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        loss   = -torch.min(surr1, surr2).mean() - entropy_coef * entropy.mean()
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.parameters(), 0.5)
        self.optimizer.step()
        return loss.item()


class MLPCritic(nn.Module):
    """Value network V(s) for continuous state observations."""

    def __init__(self, n_obs=N_OBS, hidden=HIDDEN, lr=3e-4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
        self.optimizer = optim.Adam(self.parameters(), lr=lr)

    def forward(self, obs):
        return self.net(obs).squeeze(-1)

    @torch.no_grad()
    def value(self, obs_np):
        obs_t = torch.tensor(obs_np, dtype=torch.float32).to(DEVICE)
        return self.forward(obs_t).item()

    def learn(self, obs_t, returns):
        loss = F.mse_loss(self.forward(obs_t), returns)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.parameters(), 0.5)
        self.optimizer.step()
        return loss.item()


print("MLPActor and MLPCritic defined.")

## 3. PPO Agent

Key differences from the FrozenLake version:
- **GAE** (`λ = 0.95`) replaces simple discounted returns — essential for MuJoCo
- **Larger rollout** (2048 steps) — MuJoCo episodes are long; more data per update = stable gradients
- **More k_epochs** (10) — larger batch warrants more reuse  
- **Observation normalisation** — running mean/std keeps inputs well-scaled

In [ ]:
class RunningNormalizer:
    """Online mean/variance normaliser for observations."""

    def __init__(self, shape):
        self.mean  = np.zeros(shape, dtype=np.float64)
        self.var   = np.ones(shape,  dtype=np.float64)
        self.count = 1e-4

    def update(self, x):
        batch_mean = np.mean(x, axis=0)
        batch_var  = np.var(x,  axis=0)
        batch_n    = x.shape[0]
        total      = self.count + batch_n
        delta      = batch_mean - self.mean
        self.mean  = self.mean + delta * batch_n / total
        self.var   = (self.var * self.count + batch_var * batch_n
                      + delta**2 * self.count * batch_n / total) / total
        self.count = total

    def normalise(self, x):
        return (x - self.mean) / (np.sqrt(self.var) + 1e-8)


class HumanoidPPOAgent:
    """
    PPO-Clip agent for the Humanoid-v5 MuJoCo environment.

    Parameters
    ----------
    lr             : Adam learning rate
    gamma          : discount factor
    gae_lambda     : GAE λ (0 = 1-step TD, 1 = full MC)
    epsilon        : PPO clip range
    k_epochs       : gradient updates per buffer
    rollout_length : env steps collected per epoch
    """

    def __init__(self, lr=3e-4, gamma=0.99, gae_lambda=0.95,
                 epsilon=0.2, k_epochs=10, rollout_length=2048):
        self.actor    = MLPActor().to(DEVICE)
        self.critic   = MLPCritic().to(DEVICE)
        self.norm     = RunningNormalizer((N_OBS,))
        self.gamma    = gamma
        self.lam      = gae_lambda
        self.epsilon  = epsilon
        self.k_epochs = k_epochs
        self.rollout_length = rollout_length

    # ------------------------------------------------------------------

    def _collect(self, env, obs):
        buf = []
        for _ in range(self.rollout_length):
            obs_norm = self.norm.normalise(obs)
            action, log_prob = self.actor.select_action(obs_norm)
            value            = self.critic.value(obs_norm)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            buf.append((obs_norm, action, float(reward), done, log_prob, value))
            obs = env.reset()[0] if done else next_obs
        self.norm.update(np.array([b[0] for b in buf]))
        return buf, obs

    def _gae(self, rewards, dones, values):
        advantages, gae = [], 0.0
        next_value = 0.0
        for r, d, v in zip(reversed(rewards), reversed(dones), reversed(values)):
            delta = r + self.gamma * next_value * (1 - d) - v
            gae   = delta + self.gamma * self.lam * (1 - d) * gae
            advantages.insert(0, gae)
            next_value = v
        returns = [a + v for a, v in zip(advantages, values)]
        return (torch.tensor(advantages, dtype=torch.float32),
                torch.tensor(returns,    dtype=torch.float32))

    # ------------------------------------------------------------------

    def train(self, env, n_updates=1000, seed=42):
        obs, _ = env.reset(seed=seed)
        history = {"actor_loss": [], "critic_loss": [],
                   "ep_return": [], "ep_length": []}
        ep_return, ep_len, ep_returns, ep_lens = 0.0, 0, [], []

        for update in range(1, n_updates + 1):
            buf, obs = self._collect(env, obs)

            obss      = np.array([b[0] for b in buf], dtype=np.float32)
            actions   = np.array([b[1] for b in buf], dtype=np.float32)
            rewards   = [b[2] for b in buf]
            dones     = [b[3] for b in buf]
            log_probs = torch.stack([b[4] for b in buf]).to(DEVICE)
            values    = [b[5] for b in buf]

            for r, d in zip(rewards, dones):
                ep_return += r; ep_len += 1
                if d:
                    ep_returns.append(ep_return); ep_lens.append(ep_len)
                    ep_return, ep_len = 0.0, 0

            history["ep_return"].append(np.mean(ep_returns[-5:]) if ep_returns else 0.0)
            history["ep_length"].append(np.mean(ep_lens[-5:])    if ep_lens    else 0.0)

            advantages, returns = self._gae(rewards, dones, values)
            advantages = ((advantages - advantages.mean())
                          / (advantages.std() + 1e-8)).to(DEVICE)
            returns = returns.to(DEVICE)

            obs_t = torch.tensor(obss,    dtype=torch.float32).to(DEVICE)
            act_t = torch.tensor(actions, dtype=torch.float32).to(DEVICE)

            a_losses, c_losses = [], []
            for _ in range(self.k_epochs):
                a_losses.append(self.actor.learn(obs_t, act_t, log_probs,
                                                  advantages, self.epsilon))
                c_losses.append(self.critic.learn(obs_t, returns))

            history["actor_loss"].append(sum(a_losses)  / self.k_epochs)
            history["critic_loss"].append(sum(c_losses) / self.k_epochs)

            if update % 50 == 0:
                print(f"Update {update:4d}/{n_updates}"
                      f"  ep_ret={history['ep_return'][-1]:8.1f}"
                      f"  ep_len={history['ep_length'][-1]:6.1f}"
                      f"  a_loss={history['actor_loss'][-1]:8.4f}"
                      f"  c_loss={history['critic_loss'][-1]:8.4f}")

        env.close()
        return history

    @staticmethod
    def plot(history):
        fig, axes = plt.subplots(1, 4, figsize=(20, 4))
        fig.suptitle("PPO Training — Humanoid-v5", fontsize=14, fontweight="bold")
        panels = [
            (axes[0], "ep_return",   "Episode Return (5-ep avg)", "Return", "#2ca02c"),
            (axes[1], "ep_length",   "Episode Length (5-ep avg)", "Steps",  "#9467bd"),
            (axes[2], "critic_loss", "Critic Loss (MSE)",         "Loss",   "#d62728"),
            (axes[3], "actor_loss",  "Actor Loss (PPO-Clip)",     "Loss",   "#1f77b4"),
        ]
        for ax, key, title, ylabel, color in panels:
            ax.plot(history[key], color=color, linewidth=2)
            ax.set_title(title, fontsize=11, fontweight="bold")
            ax.set_xlabel("Update Step")
            ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


print("HumanoidPPOAgent defined.")

## 4. Train

| Hyperparameter | Value | Rationale |
|----------------|:-----:|-----------|
| `rollout_length` | 2048 | Long horizon — MuJoCo episodes ≤ 1000 steps |
| `k_epochs` | 10 | More reuse from large batch |
| `lr` | 3e-4 | Standard Adam rate |
| `gamma` | 0.99 | Long-horizon discount |
| `gae_lambda` | 0.95 | Balance bias/variance |
| `epsilon` | 0.2 | PPO clip range |
| `n_updates` | 1000 | ~2M total steps |

In [ ]:
env = gym.make("Humanoid-v5", render_mode=None)

agent = HumanoidPPOAgent(
    lr=3e-4, gamma=0.99, gae_lambda=0.95,
    epsilon=0.2, k_epochs=10, rollout_length=2048,
)

history = agent.train(env, n_updates=1000, seed=42)
HumanoidPPOAgent.plot(history)